# ❤️ Heart Disease Prediction — Kaggle Playground S6E2
### XGBoost + LightGBM + Logistic Regression Ensemble · 5-Fold Stratified CV

**Yaklaşım:** Feature engineering → Label encoding → 3 model ensemble → Soft voting ile tahmin

## 1. Kütüphaneler

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

SEED = 42
N_FOLDS = 5
print('Kütüphaneler yüklendi ✓')

## 2. Veri Yükleme

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/test.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')
train.head()

## 3. Keşifsel Veri Analizi (EDA)

In [ ]:
print('=== Temel İstatistikler ===')
print(train.describe().T.to_string())

In [ ]:
print('=== Eksik Değerler ===')
missing = train.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Eksik değer yok ✓')

In [ ]:
# Hedef değişken dağılımı
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Hedef encode: string -> 0/1
train['Heart Disease'] = train['Heart Disease'].map({'Presence': 1, 'Absence': 0})

train['Heart Disease'].value_counts().plot(kind='bar', ax=axes[0], color=['#2196F3','#F44336'])
axes[0].set_title('Hedef Dağılımı')
axes[0].set_xticklabels(['Absence (0)', 'Presence (1)'], rotation=0)

# Korelasyon ısı haritası (sayısal özellikler)
num_only = train.select_dtypes(include='number')
sns.heatmap(num_only.corr(), annot=True, fmt='.2f', cmap='RdBu_r', ax=axes[1])
axes[1].set_title('Korelasyon Matrisi')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Kategorik sütun dağılımları
cat_raw = ['Chest pain type', 'FBS over 120', 'EKG results',
           'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(cat_raw):
    train.groupby(col)['Heart Disease'].mean().plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(f'{col}\nHastalık Oranı')
    axes[i].set_ylabel('Oran')
    axes[i].tick_params(axis='x', rotation=0)

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig('categorical_rates.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Özellik Mühendisliği

In [ ]:
TARGET = 'Heart Disease'
NUM_COLS = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']
CAT_COLS = ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results',
            'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium']

test_ids = test['id'].copy()

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Klinik açıdan anlamlı türetilmiş özellikler."""
    df = df.copy()

    # --- Klinik risk skoru (basit ama etkili) ---
    df['age_hr_ratio']   = df['Age'] / (df['Max HR'] + 1e-9)
    df['bp_chol_ratio']  = df['BP']  / (df['Cholesterol'] + 1e-9)
    df['hr_reserve']     = 220 - df['Age'] - df['Max HR']   # tahmini HR rezervi

    # --- Sayısal binning (quantile + equal-width) ---
    for col in NUM_COLS:
        df[f'{col}_qbin'] = pd.qcut(df[col], q=5, labels=False, duplicates='drop').fillna(0)
        df[f'{col}_bin']  = pd.cut(df[col],  bins=5, labels=False, duplicates='drop').fillna(0)

    # --- Basamak özellikleri ---
    for col in NUM_COLS:
        s = df[col].astype(int).astype(str)
        df[f'{col}_d1'] = s.str[-1].astype(int)
        df[f'{col}_d2'] = s.str.get(-2).fillna('0').astype(int)

    return df

train = build_features(train)
test  = build_features(test)

print(f'Özellik sayısı: {train.shape[1]}')
train.head()

## 5. Kategorik Encoding

In [ ]:
# id ve target'ı düşür, kategorikleri encode et
drop_cols = ['id', TARGET]
X = train.drop(columns=drop_cols, errors='ignore')
y = train[TARGET]
X_test = test.drop(columns=['id'], errors='ignore')

# Tüm object sütunları LabelEncode
le_dict = {}
for col in X.select_dtypes(include='object').columns:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col]      = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    le_dict[col] = le

print(f'X shape: {X.shape}  |  X_test shape: {X_test.shape}')

## 6. Model Eğitimi — 5-Fold Ensemble

In [ ]:
def make_models():
    """Her fold için taze model örnekleri."""
    xgb = XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=0.1, reg_lambda=1.0,
        eval_metric='auc', use_label_encoder=False,
        random_state=SEED, n_jobs=-1
    )
    lgbm = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        num_leaves=40, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1, verbose=-1
    )
    lr = LogisticRegression(max_iter=2000, C=0.5, solver='lbfgs', random_state=SEED)
    return {'xgb': xgb, 'lgbm': lgbm, 'lr': lr}


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof  = {name: np.zeros(len(X))       for name in ['xgb', 'lgbm', 'lr', 'ensemble']}
preds= {name: np.zeros(len(X_test))  for name in ['xgb', 'lgbm', 'lr', 'ensemble']}

fold_aucs = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    models = make_models()
    fold_probs = {}

    # XGBoost — early stopping
    models['xgb'].fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    fold_probs['xgb'] = models['xgb'].predict_proba(X_val)[:, 1]
    preds['xgb']     += models['xgb'].predict_proba(X_test)[:, 1] / N_FOLDS

    # LightGBM — early stopping callback
    models['lgbm'].fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )
    fold_probs['lgbm'] = models['lgbm'].predict_proba(X_val)[:, 1]
    preds['lgbm']     += models['lgbm'].predict_proba(X_test)[:, 1] / N_FOLDS

    # Logistic Regression (scaled içinde çalışır ama genelde direkt de iyidir)
    models['lr'].fit(X_tr, y_tr)
    fold_probs['lr'] = models['lr'].predict_proba(X_val)[:, 1]
    preds['lr']     += models['lr'].predict_proba(X_test)[:, 1] / N_FOLDS

    # Ensemble: ağırlıklı ortalama
    w = [0.45, 0.45, 0.10]
    ens = (w[0]*fold_probs['xgb'] + w[1]*fold_probs['lgbm'] + w[2]*fold_probs['lr'])
    oof['xgb'][val_idx]      = fold_probs['xgb']
    oof['lgbm'][val_idx]     = fold_probs['lgbm']
    oof['lr'][val_idx]       = fold_probs['lr']
    oof['ensemble'][val_idx] = ens

    preds['ensemble'] += (w[0]*preds['xgb'] + w[1]*preds['lgbm'] + w[2]*preds['lr']) / N_FOLDS

    auc = roc_auc_score(y_val, ens)
    fold_aucs.append(auc)
    print(f"Fold {fold}  |  XGB: {roc_auc_score(y_val,fold_probs['xgb']):.4f}  "
          f"LGBM: {roc_auc_score(y_val,fold_probs['lgbm']):.4f}  "
          f"LR: {roc_auc_score(y_val,fold_probs['lr']):.4f}  "
          f"Ensemble: {auc:.4f}")

print(f"\n{'='*60}")
print(f"OOF AUC — Ensemble: {roc_auc_score(y, oof['ensemble']):.4f}")
print(f"Mean Fold AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")

## 7. Değerlendirme & Görselleştirme

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC eğrisi
for name, color in [('xgb','steelblue'),('lgbm','darkorange'),('ensemble','crimson')]:
    auc = roc_auc_score(y, oof[name])
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y, oof[name])
    axes[0].plot(fpr, tpr, label=f'{name.upper()} (AUC={auc:.4f})', color=color, lw=2)
axes[0].plot([0,1],[0,1],'k--')
axes[0].set_title('ROC Eğrisi (OOF)')
axes[0].legend()

# Confusion matrix
thresh = 0.5
y_pred = (oof['ensemble'] >= thresh).astype(int)
cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title(f'Confusion Matrix (threshold={thresh})')
axes[1].set_xlabel('Tahmin'); axes[1].set_ylabel('Gerçek')

# Fold AUC bar chart
axes[2].bar(range(1, N_FOLDS+1), fold_aucs, color='teal', alpha=0.8)
axes[2].axhline(np.mean(fold_aucs), color='red', ls='--', label=f'Ort. {np.mean(fold_aucs):.4f}')
axes[2].set_title('Fold AUC Skorları')
axes[2].set_xlabel('Fold'); axes[2].set_ylabel('AUC')
axes[2].legend()

plt.tight_layout()
plt.savefig('evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n=== Classification Report (threshold=0.5) ===')
print(classification_report(y, y_pred, target_names=['Absence','Presence']))

## 8. Özellik Önemi

In [ ]:
# Son fold XGBoost ve LGBM özellik önemi
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

xgb_imp  = pd.Series(models['xgb'].feature_importances_,  index=X.columns).nlargest(20)
lgbm_imp = pd.Series(models['lgbm'].feature_importances_, index=X.columns).nlargest(20)

xgb_imp.sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('XGBoost — Top 20 Özellik')

lgbm_imp.sort_values().plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('LightGBM — Top 20 Özellik')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Submission

In [ ]:
# Ensemble tahminini kaydet
# Ensemble test_preds'i N_FOLDS ile N_FOLDS kez toplandı, düzeltelim:
final_preds = w[0]*preds['xgb'] + w[1]*preds['lgbm'] + w[2]*preds['lr']

submission = pd.DataFrame({
    'id': test_ids,
    TARGET: final_preds
})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print(f'\n✓ submission.csv kaydedildi. Shape: {submission.shape}')

---
## 10. Hugging Face İçin Model Kaydetme
Bu hücre Kaggle'da çalıştırıldıktan sonra `model_artifacts/` klasörüne gerekli dosyaları kaydeder.

In [ ]:
import joblib, os, json

os.makedirs('model_artifacts', exist_ok=True)

# Son fold modellerini ve encoderları kaydet
joblib.dump(models['xgb'],  'model_artifacts/xgb_model.pkl')
joblib.dump(models['lgbm'], 'model_artifacts/lgbm_model.pkl')
joblib.dump(models['lr'],   'model_artifacts/lr_model.pkl')
joblib.dump(le_dict,        'model_artifacts/label_encoders.pkl')

# Kolon sırasını kaydet (inference sırasında gerekli)
with open('model_artifacts/feature_columns.json', 'w') as f:
    json.dump(list(X.columns), f)

# Ensemble ağırlıklarını kaydet
with open('model_artifacts/ensemble_weights.json', 'w') as f:
    json.dump({'xgb': 0.45, 'lgbm': 0.45, 'lr': 0.10}, f)

print('✓ model_artifacts/ klasörü oluşturuldu:')
for f in os.listdir('model_artifacts'):
    print(f'   {f}')